# SpatialData Crop-to-SpatialData Core Prototype: `SLIDE-0329_crop_2048`

This notebook keeps the crop workflow intentionally linear:

- load the cropped full-merge OME-TIFF with `sopa.io.ome_tif(...)`
- load whole-cell and nuclear masks as SpatialData labels
- derive cell polygons with `spatialdata.to_polygons(...)`
- aggregate image intensities by raster cell labels
- load the Nimbus crop table and confirm exact ID agreement across all representations


## Setup and Paths


In [12]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import anndata as ad
import numpy as np
import pandas as pd
import sopa
import spatialdata
import tifffile
from shapely import make_valid
from spatialdata import SpatialData
from spatialdata.models import Labels2DModel, ShapesModel, TableModel

CROP_OUTPUTS = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs")
FULL_MERGE_PATH = CROP_OUTPUTS / "SLIDE-0329_crop_2048_full_merge.ome.tif"
CELL_MASK_PATH = CROP_OUTPUTS / "masks" / "SLIDE-0329_crop_2048_whole_cell.tiff"
NUCLEAR_MASK_PATH = CROP_OUTPUTS / "masks" / "SLIDE-0329_crop_2048_nuclear.tiff"
NIMBUS_TABLE_PATH = CROP_OUTPUTS / "nimbus" / "cell_table_full.csv"

for path in [FULL_MERGE_PATH, CELL_MASK_PATH, NUCLEAR_MASK_PATH, NIMBUS_TABLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print(
    {
        "spatialdata": spatialdata.__version__,
        "sopa": sopa.__version__,
        "tifffile": tifffile.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "anndata": ad.__version__,
    }
)
print(
    {
        "full_merge": str(FULL_MERGE_PATH),
        "cell_mask": str(CELL_MASK_PATH),
        "nuclear_mask": str(NUCLEAR_MASK_PATH),
        "nimbus_table": str(NIMBUS_TABLE_PATH),
    }
)


{'spatialdata': '0.7.2', 'sopa': '2.2.4', 'tifffile': '2026.3.3', 'pandas': '2.3.3', 'numpy': '2.4.4', 'anndata': '0.12.10'}
{'full_merge': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/SLIDE-0329_crop_2048_full_merge.ome.tif', 'cell_mask': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff', 'nuclear_mask': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff', 'nimbus_table': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/nimbus/cell_table_full.csv'}


/tmp/ipykernel_146201/3634785238.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  "anndata": ad.__version__,


## Image Load


In [13]:
image_only_sdata = sopa.io.ome_tif(FULL_MERGE_PATH)
image_key = next(iter(image_only_sdata.images))
full_image = image_only_sdata.images[image_key]
scale0 = full_image["scale0"]["image"]
channel_names = [str(value) for value in scale0.coords["c"].values[:8].tolist()]

print(
    {
        "image_key": image_key,
        "scale0_shape": tuple(scale0.shape),
        "scale0_dims": tuple(scale0.dims),
        "scale0_chunks": scale0.chunks,
        "first_channels": channel_names,
    }
)


{'image_key': 'SLIDE-0329_crop_2048_full_merge', 'scale0_shape': (24, 2048, 2048), 'scale0_dims': ('c', 'y', 'x'), 'scale0_chunks': ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (1024, 1024), (1024, 1024)), 'first_channels': ['R1_CY3_AF_I', 'R1_CY5_AF_I', 'R1_CY7_AF_I', 'R1_FITC_AF_I', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI']}


## Labels Load


In [14]:
cell_mask = tifffile.imread(CELL_MASK_PATH)
nuclear_mask = tifffile.imread(NUCLEAR_MASK_PATH)

image_canvas = tuple(int(value) for value in scale0.shape[-2:])
assert tuple(cell_mask.shape) == image_canvas, "Whole-cell mask must match the scale0 image canvas"
assert tuple(nuclear_mask.shape) == image_canvas, "Nuclear mask must match the scale0 image canvas"

cell_labels = Labels2DModel.parse(cell_mask, dims=("y", "x"))
nuclear_labels = Labels2DModel.parse(nuclear_mask, dims=("y", "x"))

base_sdata = SpatialData(
    images={"full_image": full_image},
    labels={"cell_labels": cell_labels, "nuclear_labels": nuclear_labels},
)

mask_label_ids = np.unique(cell_mask)
mask_label_ids = mask_label_ids[mask_label_ids > 0].astype(int)
mask_instance_ids = sorted(mask_label_ids.astype(str).tolist(), key=int)

print(
    {
        "cell_mask_shape": tuple(cell_mask.shape),
        "cell_mask_dtype": str(cell_mask.dtype),
        "nuclear_mask_shape": tuple(nuclear_mask.shape),
        "nuclear_mask_dtype": str(nuclear_mask.dtype),
        "nonzero_cell_label_count": len(mask_label_ids),
        "base_sdata_images": list(base_sdata.images.keys()),
        "base_sdata_labels": list(base_sdata.labels.keys()),
    }
)


{'cell_mask_shape': (2048, 2048), 'cell_mask_dtype': 'uint32', 'nuclear_mask_shape': (2048, 2048), 'nuclear_mask_dtype': 'uint32', 'nonzero_cell_label_count': 4459, 'base_sdata_images': ['full_image'], 'base_sdata_labels': ['cell_labels', 'nuclear_labels']}


## Vectorization


In [15]:
polygon_df = spatialdata.to_polygons(base_sdata.labels["cell_labels"]).copy()
polygon_df["cell_id"] = polygon_df["label"].astype(int)
polygon_df["instance_id"] = polygon_df["cell_id"].astype(str)
polygon_df = polygon_df.sort_values("cell_id").reset_index(drop=True)
polygon_df["test_category"] = [f"category_{index % 40}" for index in range(len(polygon_df))]

invalid_before = int((~polygon_df.geometry.is_valid).sum())
if invalid_before:
    invalid_mask = ~polygon_df.geometry.is_valid
    polygon_df.loc[invalid_mask, "geometry"] = polygon_df.loc[invalid_mask, "geometry"].map(make_valid)

invalid_after_make_valid = int((~polygon_df.geometry.is_valid).sum())
if invalid_after_make_valid:
    invalid_mask = ~polygon_df.geometry.is_valid
    polygon_df.loc[invalid_mask, "geometry"] = polygon_df.loc[invalid_mask, "geometry"].buffer(0)

invalid_after = int((~polygon_df.geometry.is_valid).sum())
empty_after_cleanup = int(polygon_df.geometry.is_empty.sum())
if empty_after_cleanup:
    polygon_df = polygon_df[~polygon_df.geometry.is_empty].copy()
    polygon_df = polygon_df.sort_values("cell_id").reset_index(drop=True)

cell_boundaries_df = polygon_df.set_index("instance_id", drop=False).copy()
cell_boundaries = ShapesModel.parse(cell_boundaries_df)
base_sdata["cell_boundaries"] = cell_boundaries

polygon_id_set = set(polygon_df["instance_id"].tolist())

print(
    {
        "polygon_count": len(polygon_df),
        "invalid_before": invalid_before,
        "invalid_after_make_valid": invalid_after_make_valid,
        "invalid_after": invalid_after,
        "empty_after_cleanup": empty_after_cleanup,
        "base_sdata_shapes": list(base_sdata.shapes.keys()),
        "test_categories": sorted(polygon_df["test_category"].unique().tolist()),
        "polygon_id_matches_mask_ids": polygon_id_set == set(mask_instance_ids),
    }
)

assert len(polygon_df) == len(mask_label_ids), "Expected one polygon row per nonzero cell label"
assert polygon_id_set == set(mask_instance_ids), "Polygon IDs must match the original mask labels"


{'polygon_count': 4459, 'invalid_before': 9, 'invalid_after_make_valid': 0, 'invalid_after': 0, 'empty_after_cleanup': 0, 'base_sdata_shapes': ['cell_boundaries'], 'test_categories': ['category_0', 'category_1', 'category_10', 'category_11', 'category_12', 'category_13', 'category_14', 'category_15', 'category_16', 'category_17', 'category_18', 'category_19', 'category_2', 'category_20', 'category_21', 'category_22', 'category_23', 'category_24', 'category_25', 'category_26', 'category_27', 'category_28', 'category_29', 'category_3', 'category_30', 'category_31', 'category_32', 'category_33', 'category_34', 'category_35', 'category_36', 'category_37', 'category_38', 'category_39', 'category_4', 'category_5', 'category_6', 'category_7', 'category_8', 'category_9'], 'polygon_id_matches_mask_ids': True}


## Raster Aggregation

Treat `agg_labels_table` as the canonical image-derived quantification table and attach it back onto `base_sdata`.


In [16]:
agg_sdata = spatialdata.aggregate(
    values="full_image",
    by="cell_labels",
    values_sdata=base_sdata,
    by_sdata=base_sdata,
    agg_func="mean",
    table_name="agg_labels",
)
agg_labels_table = agg_sdata.tables["agg_labels"]
base_sdata["agg_labels"] = agg_labels_table
agg_instance_ids = sorted(agg_labels_table.obs["instance_id"].astype(str).tolist(), key=int)

print(
    {
        "agg_table_shape": tuple(agg_labels_table.shape),
        "agg_row_count": agg_labels_table.n_obs,
        "agg_feature_count": agg_labels_table.n_vars,
        "base_sdata_tables": list(base_sdata.tables.keys()),
        "first_agg_features": agg_labels_table.var_names[:8].tolist(),
    }
)

assert agg_labels_table.n_obs == len(mask_label_ids), "Aggregated table row count must match nonzero cell labels"


{'agg_table_shape': (4459, 24), 'agg_row_count': 4459, 'agg_feature_count': 24, 'base_sdata_tables': ['agg_labels'], 'first_agg_features': ['channel_R1_CY3_AF_I_mean', 'channel_R1_CY5_AF_I_mean', 'channel_R1_CY7_AF_I_mean', 'channel_R1_FITC_AF_I_mean', 'channel_R1_BIT2_RS0584_CY3B_mean', 'channel_R1_BIT3_RS0015_CY5_mean', 'channel_R1_BIT4_RS0083_750_mean', 'channel_R1_DAPI_mean']}


In [17]:
base_sdata

SpatialData object
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (4459, 5) (2D shapes)
└── Tables
      └── 'agg_labels': AnnData (4459, 24)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes)

In [18]:
base_sdata['cell_boundaries']

,label,geometry,cell_id,instance_id,test_category
instance_id,,,,,
85,85,"POLYGON ((162 11.5, 161 11.5, 160 11.5, 159 11...",85,85,category_0
86,86,"POLYGON ((133 21.5, 132 21.5, 131 21.5, 130 21...",86,86,category_1
87,87,"POLYGON ((67 25.5, 66 25.5, 65 25.5, 64 25.5, ...",87,87,category_2
88,88,"MULTIPOLYGON (((101 45.5, 100 45.5, 99.5 45, 9...",88,88,category_3
89,89,"POLYGON ((31 28.5, 30.5 28, 30 27.5, 29 27.5, ...",89,89,category_4
...,...,...,...,...,...
20741,20741,"POLYGON ((2047 2047.5, 2046 2047.5, 2045 2047....",20741,20741,category_14
20748,20748,"POLYGON ((2042 2016.5, 2041 2016.5, 2040 2016....",20748,20748,category_15
20749,20749,"POLYGON ((1990 2036.5, 1989 2036.5, 1988 2036....",20749,20749,category_16


## Nimbus Table Load


In [19]:
nimbus_df = pd.read_csv(NIMBUS_TABLE_PATH)

required_columns = {"cell_id", "fov"}
missing_columns = required_columns.difference(nimbus_df.columns)
assert not missing_columns, f"Nimbus table is missing required columns: {sorted(missing_columns)}"

fov_values = sorted(nimbus_df["fov"].astype(str).unique().tolist())
assert len(fov_values) == 1, f"Expected exactly one FOV, found {fov_values}"
assert nimbus_df["cell_id"].is_unique, "Nimbus cell_id values must be unique"

feature_columns = [column for column in nimbus_df.columns if column not in {"cell_id", "fov"}]
numeric_feature_columns = [column for column in feature_columns if pd.api.types.is_numeric_dtype(nimbus_df[column])]
assert len(feature_columns) == len(numeric_feature_columns), "All Nimbus feature columns must be numeric"
assert len(numeric_feature_columns) == 4, f"Expected four numeric Nimbus feature columns, found {numeric_feature_columns}"

nimbus_df = nimbus_df.sort_values("cell_id").reset_index(drop=True)
nimbus_obs = nimbus_df[["cell_id", "fov"]].copy()
nimbus_obs["instance_id"] = nimbus_obs["cell_id"].astype(str)
nimbus_obs["region"] = "cell_labels"
nimbus_obs.index = nimbus_obs["instance_id"]

nimbus_adata = ad.AnnData(
    X=nimbus_df[numeric_feature_columns].to_numpy(dtype=float),
    obs=nimbus_obs,
    var=pd.DataFrame(index=pd.Index(numeric_feature_columns, name="feature")),
)
nimbus_table = TableModel.parse(
    nimbus_adata,
    region="cell_labels",
    region_key="region",
    instance_key="instance_id",
)

base_sdata["nimbus_table"] = nimbus_table

nimbus_instance_ids = sorted(base_sdata.tables["nimbus_table"].obs["instance_id"].astype(str).tolist(), key=int)

print(
    {
        "nimbus_shape": tuple(nimbus_df.shape),
        "nimbus_fov": fov_values[0],
        "nimbus_feature_columns": numeric_feature_columns,
        "nimbus_row_count": base_sdata.tables["nimbus_table"].n_obs,
        "base_sdata_tables": list(base_sdata.tables.keys()),
    }
)


{'nimbus_shape': (4459, 6), 'nimbus_fov': 'SLIDE-0329_crop_2048', 'nimbus_feature_columns': ['SLIDE-0329_1.0.4_R000_DAPI__FINAL_F', 'SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F'], 'nimbus_row_count': 4459, 'base_sdata_tables': ['agg_labels', 'nimbus_table']}


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


In [20]:
base_sdata['cell_boundaries']

,label,geometry,cell_id,instance_id,test_category
instance_id,,,,,
85,85,"POLYGON ((162 11.5, 161 11.5, 160 11.5, 159 11...",85,85,category_0
86,86,"POLYGON ((133 21.5, 132 21.5, 131 21.5, 130 21...",86,86,category_1
87,87,"POLYGON ((67 25.5, 66 25.5, 65 25.5, 64 25.5, ...",87,87,category_2
88,88,"MULTIPOLYGON (((101 45.5, 100 45.5, 99.5 45, 9...",88,88,category_3
89,89,"POLYGON ((31 28.5, 30.5 28, 30 27.5, 29 27.5, ...",89,89,category_4
...,...,...,...,...,...
20741,20741,"POLYGON ((2047 2047.5, 2046 2047.5, 2045 2047....",20741,20741,category_14
20748,20748,"POLYGON ((2042 2016.5, 2041 2016.5, 2040 2016....",20748,20748,category_15
20749,20749,"POLYGON ((1990 2036.5, 1989 2036.5, 1988 2036....",20749,20749,category_16


## Quick Consistency Checks


In [21]:
mask_id_set = set(mask_instance_ids)
polygon_id_set = set(polygon_df["instance_id"].tolist())
agg_id_set = set(agg_labels_table.obs["instance_id"].astype(str).tolist())
nimbus_id_set = set(base_sdata.tables["nimbus_table"].obs["instance_id"].astype(str).tolist())

print(
    {
        "mask_count": len(mask_id_set),
        "polygon_count": len(polygon_id_set),
        "agg_count": len(agg_id_set),
        "nimbus_count": len(nimbus_id_set),
    }
)
print("First 10 mask IDs:", sorted(mask_id_set, key=int)[:10])
print("First 10 polygon IDs:", sorted(polygon_id_set, key=int)[:10])
print("First 10 aggregate IDs:", sorted(agg_id_set, key=int)[:10])
print("First 10 Nimbus IDs:", sorted(nimbus_id_set, key=int)[:10])
print("mask - polygons:", sorted(mask_id_set - polygon_id_set, key=int)[:10])
print("polygons - mask:", sorted(polygon_id_set - mask_id_set, key=int)[:10])
print("mask - aggregate:", sorted(mask_id_set - agg_id_set, key=int)[:10])
print("aggregate - mask:", sorted(agg_id_set - mask_id_set, key=int)[:10])
print("mask - Nimbus:", sorted(mask_id_set - nimbus_id_set, key=int)[:10])
print("Nimbus - mask:", sorted(nimbus_id_set - mask_id_set, key=int)[:10])

assert mask_id_set == polygon_id_set == agg_id_set == nimbus_id_set, "All four ID sets must match exactly"

comparison_df = pd.DataFrame(
    {
        "mask_instance_id": sorted(mask_id_set, key=int),
        "polygon_instance_id": sorted(polygon_id_set, key=int),
        "agg_instance_id": sorted(agg_id_set, key=int),
        "nimbus_instance_id": sorted(nimbus_id_set, key=int),
    }
)
comparison_df["all_match"] = comparison_df.nunique(axis=1) == 1
print(comparison_df.head(10).to_string(index=False))

assert comparison_df["all_match"].all(), "Sorted ID rows must agree across all four views"
print("Exact ID agreement confirmed across masks, polygons, aggregation, and Nimbus.")


{'mask_count': 4459, 'polygon_count': 4459, 'agg_count': 4459, 'nimbus_count': 4459}
First 10 mask IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']
First 10 polygon IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']
First 10 aggregate IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']
First 10 Nimbus IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']
mask - polygons: []
polygons - mask: []
mask - aggregate: []
aggregate - mask: []
mask - Nimbus: []
Nimbus - mask: []
mask_instance_id polygon_instance_id agg_instance_id nimbus_instance_id  all_match
              85                  85              85                 85       True
              86                  86              86                 86       True
              87                  87              87                 87       True
              88                  88              88                 88       True
              89                  89              89

In [22]:
base_sdata.write("test.zarr")

/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(
